In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/email-Enron")
PREFIX = "email-Enron"


In [ ]:
def load_hypergraph(data_dir: Path, prefix: str):
    """Read a timestamped simplex dataset (prefix-nverts.txt, prefix-simplices.txt,
    prefix-times.txt) and return (hyperedges, times), where each hyperedge is a
    sorted tuple of node ids."""
    nverts = np.loadtxt(data_dir / f"{prefix}-nverts.txt", dtype=int)
    simplices = np.loadtxt(data_dir / f"{prefix}-simplices.txt", dtype=int)
    times = np.loadtxt(data_dir / f"{prefix}-times.txt", dtype=int)

    nverts = np.atleast_1d(nverts)
    simplices = np.atleast_1d(simplices)
    times = np.atleast_1d(times)

    if nverts.sum() != len(simplices):
        raise ValueError("Malformed input: sum(nverts) != len(simplices)")
    if len(nverts) != len(times):
        raise ValueError("Malformed input: len(nverts) != len(times)")

    hyperedges = []
    pos = 0
    for c in nverts:
        nodes = simplices[pos : pos + c]
        hyperedges.append(tuple(sorted(map(int, nodes))))
        pos += c

    return hyperedges, times


In [ ]:
hyperedges, times = load_hypergraph(DATA_DIR, PREFIX)

# Keep only 3-uniform hyperedges
edges3 = []
times3 = []

for e, t in zip(hyperedges, times):
    if len(e) == 3:
        edges3.append(e)
        times3.append(int(t))

print(f"Total hyperedges: {len(hyperedges)}")
print(f"Order-3 hyperedges: {len(edges3)}")

# Keep only earliest occurrence of each hyperedge
earliest = {}
for e, t in zip(edges3, times3):
    if e not in earliest or t < earliest[e]:
        earliest[e] = t

edges3 = sorted(earliest.keys())
times3 = [earliest[e] for e in edges3]

print(f"Unique order-3 hyperedges: {len(edges3)}")


In [ ]:
df = pd.DataFrame(edges3, columns=["i", "j", "k"])
out_path = DATA_DIR / "enron_order3_edges.csv"
df.to_csv(out_path, index=False)
out_path


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

edges = pd.read_csv("../data/email-Enron/enron_order3_edges.csv")


In [ ]:
# Stack all node IDs into one vector
nodes = pd.concat([edges["i"], edges["j"], edges["k"]])

# Degree = number of hyperedges containing the node
degree = nodes.value_counts().sort_index()

degree.head()


In [ ]:
plt.figure(figsize=(5,4))
plt.hist(degree.values, bins=50)
plt.xlabel("Degree")
plt.ylabel("Number of nodes")
plt.title("Degree distribution (3-uniform Enron hypergraph)")
plt.tight_layout()
plt.show()
